In [1]:
import pandas as pd

train_df = pd.read_parquet('../data/train.parquet')
test_df = pd.read_parquet('../data/test.parquet')

feature_cols = [c for c in train_df.columns if c not in ['visit', 'treatment_flag', 'spend']]

X_train, y_train, t_train, spend_train = train_df[feature_cols], train_df['visit'], train_df['treatment_flag'], train_df['spend']
X_test, y_test, t_test, spend_test = test_df[feature_cols], test_df['visit'], test_df['treatment_flag'], test_df['spend']

print(X_train.shape, X_test.shape)
print(feature_cols)

(29885, 11) (12808, 11)
['recency', 'history', 'mens', 'womens', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web', 'zip_code_Rural', 'zip_code_Surburban', 'zip_code_Urban']


In [2]:
from xgboost import XGBClassifier

X_train_with_t = X_train.assign(treatment=t_train)
X_test_with_t = X_test.assign(treatment=t_test)

baseline = XGBClassifier(random_state=42, eval_metric='logloss').fit(X_train_with_t, y_train)

In [3]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

y_pred_proba = baseline.predict_proba(X_test_with_t)[:, 1]
y_pred = baseline.predict(X_test_with_t)

print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1:        {f1_score(y_test, y_pred):.4f}")

importances = pd.Series(baseline.feature_importances_, index=X_train_with_t.columns).sort_values(ascending=False)
print("\nFeature importances:")
print(importances)

ROC-AUC:   0.6152
Precision: 0.2410
Recall:    0.0122
F1:        0.0232

Feature importances:
womens                  0.152712
newbie                  0.147527
treatment               0.100344
zip_code_Rural          0.088610
mens                    0.077780
channel_Web             0.077046
recency                 0.067513
history                 0.062847
channel_Phone           0.061813
channel_Multichannel    0.058513
zip_code_Surburban      0.052680
zip_code_Urban          0.052616
dtype: float32


The model predicts who is likely to visit the site, it is **not** an uplift number. ROC-AUC of 0.615 confirms the features carry real signal (well above 0.5, nowhere near the 0.95+ that would suggest a leak), but precision (0.24) and recall (0.01) at the default 0.5 threshold are close to meaningless here; with an ~13% base visit rate and no class-weighting, the model rarely crosses 0.5 for the positive class.

Feature importances put `womens` and `newbie` ahead of `treatment` itself, with `treatment` still a solid third (0.100), the model is picking up the campaign's effect at the individual level, consistent with previously seen aggregate lift, but customer behavior explains more of who visits than the email alone does. This still cannot say who visits *because of* the email versus who would have visited regardless.

**Leakage check** The one-hot encoding in the last file was fit on the full dataset before the train/test split, not train-only. This is stated as safe there because one-hot encoding of fixed categories (`channel`, `zip_code`) doesn't depend on the outcome, unlike target encoding or scaling, so no target information crossed from test into train. Nothing in this notebook touched `X_test` before the model was fit: `X_test_with_t` was only built for evaluation, after `baseline` was already trained on `X_train_with_t` alone.